In [1]:
suppressMessages({
    library(dplyr)
    library(parallel)
    library(ggplot2)
    library(tidyr)
        library(jsonlite)
    library(hydroGOF)
    })

In [2]:
gauge_df=readRDS('/nas/cee-water/cjgleason/colin/analyze confluence runs/SVS_df.rds')
gauged_reaches=unique(gauge_df$reach_id)
# swot_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/swot/'
# sos_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/sos/'
swot_base='/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/swot/'
sos_base='/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/sos/'

reach_ids=gauged_reaches[gauged_reaches %in% substr(list.files(swot_base),1,11)]

In [3]:
Q_prior='monthly'
output_path='/nas/cee-water/cjgleason/colin/BUSBOI/debug tests/final_monthly/'
run_ids=substr(list.files(output_path),1,11)
unrun= reach_ids[!reach_ids %in% run_ids]
length(unrun)

[1] 24

In [4]:
SVS_df= readRDS('/nas/cee-water/cjgleason/colin/analyze confluence runs/SVS_df.rds')
#now, throw out the training gauges
training_IDs='/nas/cee-water/cjgleason/colin/analyze confluence runs/reachids_svs_gages_used_to_train.json'
training_df=data.frame(training_ID=as.character(fromJSON(training_IDs)))

SVS_df= filter(SVS_df,!(reach_id %in% training_df$training_ID))
SVS_reaches=unique(SVS_df$reach_id)


In [5]:
#get reaches in the SVS and BUSBOI
length(run_ids %in% SVS_reaches)

[1] 2170

In [14]:
 source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')
suppressWarnings({
unrun=reach_ids
for (i in 2:36){
test= main_function(unrun[i],
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                   Q_prior='daily', #'daily' or 'monthly'
                   tulip='OFF', #'ON' or 'OFF'
                   GVF_on=0, # 0 or 1
                   fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
    
}

})

# a= Sys.time()
# clust=makeCluster(4)
# test=parLapply(clust,unrun[1:80],main_function,
#                    output_path=output_path,
#                    swot_base=swot_base,
#                    sos_base=sos_base,
#                    Q_prior='monthly', #or 'monthly'
#                    tulip='OFF', #or 'OFF'
#                    GVF_on=0,
#                    fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
# stopCluster(clust)
# print(Sys.time()-a)

ERROR: Error in FUN(newX[, i], ...): object 'bonk' not found


In [39]:
###to control where the slurm files are written
working_dir='/nas/cee-water/cjgleason/colin/BUSBOI/debug_logs/'
setwd(working_dir)

source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')

library(rslurm, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)
library(whisker, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)

testname=Q_prior
#slurm block
slurm_options= list(mem=64000, 'time'='50:00:00', #options for memory, time, parition, and an error file
                    partition ='ceewater_cjgleason-cpu',
                    error='slurm-%A_%a.err')
                    # nodelist = 'ceewater-cpu008')
                    # exclude='ceewater-cpu009')
sjob <- slurm_map(as.list(unrun), #thing you want to loop over. must be a list
                  main_function,  # name of the function. declared above with the 'source' command
                  jobname = testname, # defined above. just for convenience
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                  Q_prior=Q_prior,
                  tulip='OFF',
                  GVF_on=0,
                  fix_bed=0,  #0 = 5pts, 1= 1pt, 2 = fixed
                  nodes = 1, # many nodes do you want?
                  preschedule_cores=FALSE, # keep this FALSE
                  cpus_per_node = 30, # how many CPUS per node. 
                  submit = TRUE, # if TRUE, submits to the cluster
                  slurm_options=slurm_options, #defined above
                  libPaths="/nas/cee-water/cjgleason/r-lib/" ) #library paths

Submitted batch job 56689436

